In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
dataset = "/content/drive/MyDrive/Yoga Pose/Dataset"

In [3]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import Accuracy
import numpy as np

In [4]:
xcep = Xception(include_top=False, weights="imagenet", input_shape=[224,224,3])

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
for layer in xcep.layers:
  layer.trainable = False

In [6]:
x = Flatten()(xcep.output)

In [7]:
prediction = Dense(5, activation='softmax')(x)

In [8]:
model = Model (inputs = xcep.input, outputs=prediction)

In [9]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 55, 55,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add[0][0]       

 Total params: 21,363,245 (81.49 MB)

 Trainable params: 501,765 (1.91 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [10]:
model.compile(optimizer='adam',
              loss = 'categorical_crossentropy',
              metrics = ['accuracy'])

In [11]:
#import image datagenerator library
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
data = ImageDataGenerator(horizontal_flip=True, shear_range=0.2, zoom_range=0.2, rescale=1./255, validation_split=0.3)

In [13]:
training_set = data.flow_from_directory(dataset,
                                        target_size=(224,224),
                                        batch_size=16,
                                        subset='training',
                                        class_mode='categorical')

Found 694 images belonging to 5 classes.


In [14]:
testing_set = data.flow_from_directory(dataset,
                                       target_size=(224,224),
                                       batch_size=16,
                                       subset='validation',
                                       class_mode='categorical')

Found 294 images belonging to 5 classes.


In [15]:
len(training_set), len(testing_set)

(44, 19)

In [16]:
model.fit(training_set,
          validation_data = testing_set,
          epochs =26,
          steps_per_epoch=len (training_set)//16,
          validation_steps = len(testing_set)//16)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 31s 21s/step - accuracy: 0.2292 - loss: 6.1320 - val_accuracy: 0.6250 - val_loss: 2.2474
Epoch 2/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 17s/step - accuracy: 0.6042 - loss: 3.2031 - val_accuracy: 0.6250 - val_loss: 3.6203
Epoch 3/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 11s/step - accuracy: 0.3958 - loss: 7.4864 - val_accuracy: 0.7500 - val_loss: 2.2278
Epoch 4/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 17s 14s/step - accuracy: 0.7292 - loss: 2.1639 - val_accuracy: 0.8750 - val_loss: 0.1742
Epoch 5/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 13s 10s/step - accuracy: 0.7917 - loss: 1.2647 - val_accuracy: 0.7500 - val_loss: 1.7608
Epoch 6/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 13s 10s/step - accuracy: 0.8125 - loss: 1.3383 - val_accuracy: 0.8125 - val_loss: 1.0241
Epoch 7/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 13s 10s/step - accuracy: 0.7500 - loss: 1.2339 - val_accuracy: 0.9375 - val_loss: 0.5543
Epoch 8/26
2/2 ━━━━━━━━━━━━━━━━━━━━ 16s 12s/step - accuracy: 0.8333 - loss: 0.4843 - val_accuracy: 0.6875 - val_loss: 3.1538


In [17]:
model.save('xcep_yoga.h5')

In [18]:
from tensorflow.keras.models import load_model
model = load_model('xcep_yoga.h5')

In [19]:
img = load_img("/content/drive/MyDrive/Yoga Pose/Dataset/Plank/00000000.jpg", target_size=(224,224))

In [20]:
img = image.img_to_array(img)

In [21]:
img.shape

(224, 224, 3)

In [22]:
x = np.expand_dims (img, axis=0)
img_data=preprocess_input(x)
img_data.shape

(1, 224, 224, 3)

In [23]:
pred = model.predict(img_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [24]:
p = np.argmax(pred)

In [25]:
columns = ['Downdog', 'Goddess', 'Plank', 'Tree', 'Warrior2']

In [26]:
result = str(columns[p])

In [27]:
result

'Plank'

In [28]:
from google.colab import files
files.download('xcep_yoga.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>